In [26]:
import re
import numpy as np
from math import log

## Build dataset

In [27]:
train_data = [
    ("I love this movie", 1),
    ("This film is amazing", 1),
    ("I hate this movie", 0),
    ("This film is terrible", 0)
]


comvention:
- 1 → positve
- 0 → negative

In [28]:
for sentence, label in train_data:
    print(f'Sentence: {sentence}')
    print(f'label: {label}')
    print()

Sentence: I love this movie
label: 1

Sentence: This film is amazing
label: 1

Sentence: I hate this movie
label: 0

Sentence: This film is terrible
label: 0



In [29]:
def compute_priors (train_data):
    """
    args: train_data: list of tuples of sentences and their label 
    return: positive prior, negative prior
    """
    
    positive_count = 0
    negative_count = 0

    total_sentence = len (train_data)

    for sentence, label in train_data:
        if label == 1:
            positive_count +=1
        else:
            negative_count +=1

    p_pos = positive_count/total_sentence
    p_neg = negative_count/total_sentence

    return p_pos, p_neg


In [30]:
p_positive,p_negative = compute_priors(train_data)

print("positive prior", p_positive)
print("negative prior", p_negative)

positive prior 0.5
negative prior 0.5


Current model only knows:
- how common positive reviews are
- how common negative reviews are 

## Preprocess text
The preprocessing pipeline usually includes:
- lowercase conversion
- punctuation removal
- tokenization
- optional stop word removal
- optional stemming

In [31]:
def preprocess (text):
    """
    args: text 
    return: tokens 
    """
    text = text.lower()

    text = re.sub(r'[^a-zA-Z\s]', '', text)

    tokens = text.split()
    
    return tokens

In [32]:
train_data = [
    ("I love this movie", 1),
    ("This film is amazing", 1),
    ("What a great experience", 1),
    ("I really enjoyed this", 1),

    ("I hate this movie", 0),
    ("This film is terrible", 0),
    ("What a bad experience", 0),
    ("I really dislike this", 0)
]

In [33]:
train_data[1]

('This film is amazing', 1)

In [34]:
def build_freqs(train_data):
    """
    args: train_data: list of tuples of sentences and their label 
    return: freqs: dict which key is tuple of tokens and their label, 
    and value is the frequency of appearing of the key 
    """    
    freqs = {}

    for tub in train_data:
        
        sentence = tub[0]        
        label = tub[1]
        
        tokens = preprocess (sentence)
        for token in tokens:
            pair = (token,label) # set  -> membership / uniqueness

            if pair in freqs:
                freqs[(pair)] += 1 # dict -> mapping from thing to value
            
            else:
                freqs[(pair)] = 1
            

    return(freqs)



In [35]:
freqs = {
    ("love", 1): 1,
    ("movie", 1): 1,
    ("movie", 0): 1,
    ("terrible", 0): 1
}

In [36]:
def build_vocab(freqs):
    """
    args: freqs
    return: vocab: a set of all unique words in the freqs  
    """
    pairs = freqs.keys()
    vocab = set ()
    
    for pair in pairs:
        vocab.add(pair [0])
    
    return (vocab)

In [37]:
print(build_vocab(freqs))

{'movie', 'love', 'terrible'}


## Convert text into statistical information model can use

$$P(word ∣ class) = \frac{\text{count ( word, class)} ​}{\text{total words in class}}$$

- After smoothing:
$$P(word ∣ class) = \frac{\text{count ( word, class)} ​ + 1​ }{\text{total words in class} + V}$$

where V is the vocabulary size.

The +1 prevents zero probability.


In [38]:
freqs = {
    ("love", 1): 2,
    ("movie", 1): 3,
    ("great", 1): 1,

    ("hate", 0): 2,
    ("movie", 0): 1,
    ("bad", 0): 3
}

n_pos = 6   # 2 + 3 + 1
n_neg = 6   # 2 + 1 + 3

vocab = {"love", "movie", "great", "hate", "bad"}
V = len(vocab)

In [39]:
word_count = freqs.get(("love", 1), 0)
word_count

2

In [40]:
def count_words_by_label(freqs):
    """
    args: freqs
    return: n_pos: total positive words occurrence
    return: n_neg: total negative words occurrence
    """

    n_pos = 0
    n_neg = 0


    for (word, label), count in freqs.items(): 
        # we need total word occurrences, so we add the frequency value itself.
        if label == 1:
            n_pos += count 

        else:
            n_neg += count

    return n_pos, n_neg

In [41]:
print(count_words_by_label(freqs))

(6, 6)


In [42]:
def get_word_probability(freqs, word, label, n_pos, n_neg, V):
    """
    args: freqs
        word: the target word
        label 
        n_pos: the total number of positive labels
        n_neg: the total number of negative labels
        V: The number of words in the vocab
    return: the probability of the word
    """
    
    word_count = freqs.get((word, label), 0)
    if label == 1:
        denominator = n_pos + V
        
    else:
        denominator = n_neg + V
        
    prob = (word_count + 1) / (denominator)

    return prob

**The probability of sentence**

$$ P(sentence∣positive) = P(w1​∣positive)×P(w2​∣positive)×...×P(wn​∣positive)  $$

Multiplying many tiny probabilities can become extremely small. So in real implementations, we usually use log and add instead of multiply.

The solution is the logarithm identity:
$$ log(a×b)=log(a)+log(b) $$

We compute:
$$ log(P(positive))+log(P(w1​∣positive))+log(P(w2​∣positive))+... $$


In [43]:
def score_sentence(sentence, label, freqs, vocab, n_pos, n_neg, p_pos, p_neg):
    """
    args: sentence
        label 
        freqs
        Vocab: The vocabulary of words we have from the training data
        n_pos: the total number of positive labels
        n_neg: the total number of negative labels
        p_pos: positive prior
        p_neg: negative prior
    return: score: the log probability of sentence given the label whither it is positive or negative 
        
    """
    if label == 1:
        score = log(p_pos)
    else:
        score = log(p_neg) 

    
    tokens = preprocess(sentence)
    
    for token in tokens:
        if token in vocab: # means: ignore words the model never saw during training.
            prob = get_word_probability(freqs, token, label, n_pos, n_neg, len(vocab))
            score += log(prob)
            

    return score
    
    

In [44]:
score_sentence(sentence, label, freqs, vocab, n_pos, n_neg, p_positive, p_negative)

-0.6931471805599453

In [45]:
def train_naive_bayes(train_data):
    
    freqs = build_freqs(train_data)
    
    vocab = build_vocab(freqs)
    
    n_pos, n_neg =  count_words_by_label(freqs)

    p_pos, p_neg = compute_priors(train_data)

    return freqs, vocab, n_pos, n_neg, p_pos, p_neg

In [46]:
freqs, vocab, n_pos, n_neg, p_pos, p_neg = train_naive_bayes(train_data)

print(p_pos, p_neg)
print(n_pos, n_neg)
print(vocab)

0.5 0.5
16 16
{'dislike', 'film', 'amazing', 'hate', 'is', 'this', 'really', 'movie', 'a', 'great', 'i', 'experience', 'enjoyed', 'bad', 'love', 'what', 'terrible'}


In [51]:
def predict (sentence, model):
    
    freqs, vocab, n_pos, n_neg, p_pos, p_neg = model

    positive_score = score_sentence (sentence, 1, freqs, vocab, n_pos, n_neg, p_pos, p_neg) 
    negative_score = score_sentence (sentence, 0, freqs, vocab, n_pos, n_neg, p_pos, p_neg) 

    if positive_score > negative_score: # The larger log score wins. Because -5 > -10 and -5 is less negative, so it represents a higher probability.
        prediction  = "positive"

    else:
        prediction  = "negative"

    if abs(positive_score - negative_score) < 1e-9: # when the difference between the scores is so small make the prediction neutral
        prediction = "neutral/uncertain"


    return prediction, positive_score, negative_score


In [52]:
model = train_naive_bayes(train_data)

print (predict("I love this movie", model),

predict("This film is amazing", model),

predict("I hate this movie", model),

predict("This is terrible", model))

('positive', -10.807976415517974, -11.50112359607792) ('positive', -11.21344152362614, -11.906588704186085) ('negative', -11.50112359607792, -10.807976415517974) ('negative', -9.10322832327955, -8.410081142719605)


In [53]:
print (predict("great movie", model),

predict("bad film", model),

predict("love and hate", model))

('positive', -6.299867942373015, -6.99301512293296) ('negative', -6.99301512293296, -6.299867942373015) ('neutral/uncertain', -6.9930151229329605, -6.99301512293296)
